# 07 — Treino de uma arquitetura por vez (backbone + fusão à sua escolha)

Treina **uma combinação** de backbone visual/textual + mecanismo de fusão,
nas 10 tarefas do `libero_object` (mesmo escopo de
`configs/libero_object_language.py`). Ver `docs/hdf5_migration.md` e o
histórico de commits da ablação de "backbones pareados" (`src/act_lang/models/backbones/`)
para o raciocínio completo de por que CLIP/SigLIP2 entraram como
alternativa ao ResNet18+MiniLM.

**Você só escolhe duas coisas** (seção 2): `BACKBONE_TYPE` e `FUSION_TYPE`.
Tudo relacionado a otimização de treino já vem embutido no config
(`make_config`, em `configs/libero_object_language.py`), não precisa
configurar na mão:
- **Congelamento**: `freeze_vision_backbone` é `True` automaticamente pra
  qualquer backbone que não seja `resnet18` (CLIP/SigLIP são ~8x maiores e
  pré-treinados em escala muito maior -- fine-tuning completo com o pouco
  dado do LIBERO arriscaria destruir a representação pré-treinada).
- **Grupos de LR**: `training/optim.py` já separa `lr_backbone` (10x menor)
  de `lr` normal, com o prefixo generalizado pra cobrir `CLIPVisionBackbone`/
  `SiglipVisionBackbone` (não só o `VisionBackbone` original).
- **Texto pareado**: escolher `BACKBONE_TYPE="clip_vitb32"` já troca o
  encoder de texto pra torre de texto do PRÓPRIO CLIP (não MiniLM) --
  `scripts/train.py::build_model_and_optimizer` resolve isso sozinho via
  `resolve_text_encoder_spec`.

Precisa do extra `vlm` (`transformers` + `sentencepiece`) além de `hdf5`/`language`
-- só relevante mesmo se `BACKBONE_TYPE` != `"resnet18"`, mas instalado
sempre aqui por simplicidade.

**Primeiro treino, validação rápida**: `TASK_LIMIT` (seção 3) deixa baixar
só um punhado das 10 tarefas do `libero_object` em vez das 10 completas --
comece pequeno (`TASK_LIMIT=2`) pra confirmar que a combinação de
backbone+fusão escolhida treina de ponta a ponta antes de comprometer o
dataset completo e horas de GPU.

## 1. Repositório e ambiente

In [ ]:
!git clone -b teste https://github.com/rafaelheydt/act-lang.git
%cd act-lang

!pip install -q -e ".[hdf5,language,vlm]"

# CORREÇÃO: `pip install -e .` só é lido por PRÓXIMOS interpretadores Python
# (via .pth em site-packages) -- o kernel deste notebook, já rodando, não
# releria sozinho, e `import act_lang`/`import configs` falhariam com
# ModuleNotFoundError até um Runtime > Restart Session manual. Inserir os
# caminhos direto no sys.path deste processo evita depender disso.
import sys
sys.path.insert(0, "src")
sys.path.insert(0, ".")
sys.path.insert(0, "scripts")  # pra importar train.py como módulo na seção 6 (curvas)

## 2. Escolha de arquitetura

`BACKBONE_TYPE`: `"resnet18"` (original, MiniLM pro texto) |
`"clip_vitb32"` | `"siglip2_base"` (torre de texto pareada com a de
imagem, ambas do mesmo checkpoint).

`FUSION_TYPE`: `"token"` | `"film"` | `"cross_attn"`.

In [ ]:
BACKBONE_TYPE = "clip_vitb32"  # "resnet18" | "clip_vitb32" | "siglip2_base"
FUSION_TYPE = "film"           # "token" | "film" | "cross_attn"

# Mapeia (backbone, fusão) -> nome de --config (CONFIG_REGISTRY, scripts/train.py)
# e -> (módulo, atributo) do dict de config, pra seção 6 (curvas) carregar
# o mesmo cfg sem reconstruir a lógica de make_config na mão.
_CLI_CONFIG_NAME = {
    ("resnet18", "token"): "language_token",
    ("resnet18", "film"): "language_film",
    ("resnet18", "cross_attn"): "language_cross_attn",
    ("clip_vitb32", "film"): "language_clip_film",
    ("clip_vitb32", "token"): "language_clip_token",
    ("clip_vitb32", "cross_attn"): "language_clip_cross_attn",
    ("siglip2_base", "film"): "language_siglip_film",
    ("siglip2_base", "token"): "language_siglip_token",
    ("siglip2_base", "cross_attn"): "language_siglip_cross_attn",
}
CLI_CONFIG_NAME = _CLI_CONFIG_NAME[(BACKBONE_TYPE, FUSION_TYPE)]

import train as train_script  # scripts/train.py como módulo -- reaproveita load_config/CONFIG_REGISTRY

cfg = train_script.load_config(CLI_CONFIG_NAME)
print(f"--config {CLI_CONFIG_NAME!r} -> experiment_name={cfg['experiment_name']!r}")
print(f"backbone_type={cfg.get('backbone_type', 'resnet18')!r} | fusion_type={cfg['fusion_type']!r} "
      f"| freeze_vision_backbone={cfg.get('freeze_vision_backbone', False)}")

## 3. Baixar o dataset (tarefas do `libero_object`)

Baixa por suíte oficial (`--suite libero_object`), não por `PILOT_LIMIT`
solto (que pegaria as N primeiras em ordem alfabética das 40, não
necessariamente dentro dessa suíte). Vai pro disco local do Colab
(`/content`), não pro Drive -- ver `docs/hdf5_migration.md` sobre por quê.

`TASK_LIMIT`: `None` baixa as 10 tarefas completas do `libero_object`;
um número baixa só as N primeiras (ordem alfabética, determinística) DENTRO
da suíte -- útil pra um primeiro treino de validação rápida (poucos
cenários) antes de comprometer as 10 completas. `HDF5LiberoDataset` já
treina automaticamente só nas tarefas que estiverem presentes em `DATA_DIR`
(interseção com `cfg["task_texts"]`), então não precisa mexer no config --
baixar menos tarefas já basta.

In [ ]:
DATA_DIR = "/content/libero_hdf5_object"
TASK_LIMIT = 2  # None = as 10 tarefas completas do libero_object; comece pequeno pra validar

import subprocess
cmd = [sys.executable, "-u", "scripts/download_libero_hdf5.py", "--out", DATA_DIR, "--suite", "libero_object"]
if TASK_LIMIT is not None:
    cmd += ["--limit", str(TASK_LIMIT)]
print("rodando:", " ".join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
assert proc.returncode == 0, "download falhou -- veja o log acima"

## 4. Treino

Monta o Drive só pros **checkpoints** (o dataset fica no disco local,
seção 3). `CHECKPOINT_DIR` usa o sufixo `_hdf5128` -- mesma convenção do
`04_treino_hdf5.ipynb`, evita colidir com um checkpoint do mesmo
`experiment_name` treinado no pipeline JPEG 256×256 antigo (só relevante
pra `BACKBONE_TYPE="resnet18"`, já que CLIP/SigLIP têm `experiment_name`
próprio -- mas aplicado sempre, por simplicidade e consistência).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime
CHECKPOINT_DIR = f"/content/drive/MyDrive/{cfg['experiment_name']}_hdf5128"
print("checkpoints em:", CHECKPOINT_DIR)

Path("logs").mkdir(exist_ok=True)
log_path = f"logs/treino_{BACKBONE_TYPE}_{FUSION_TYPE}_{datetime.now():%Y%m%d_%H%M}.log"

cmd = [
    sys.executable, "-u", "scripts/train.py",
    "--config", CLI_CONFIG_NAME,
    "--hdf5-dir", DATA_DIR,
    "--checkpoint-dir", CHECKPOINT_DIR,
    "--resume",
]
print("rodando:", " ".join(cmd))
print("log em:", log_path, "\n" + "=" * 60)

with open(log_path, "a") as logf:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
        logf.write(line)
        logf.flush()
    proc.wait()

print("\n" + "=" * 60)
print(f"processo encerrado com código {proc.returncode}"
      + (" -- verifique o log acima" if proc.returncode != 0 else " -- ok"))

## 5. Curvas

Reaproveita `train_script.build_model_and_optimizer(cfg, device)` (em vez
de reconstruir `ACT(...)` na mão) -- constrói o backbone/fusão certos
automaticamente a partir do MESMO `cfg` usado no treino, sem duplicar a
lógica de qual text encoder pareia com qual backbone. Baixa o checkpoint
pré-treinado do backbone de novo só pra ter um objeto com a arquitetura
certa pra carregar o `state_dict` em cima -- rápido (a torre já fica em
cache do HF depois da primeira vez) e o resultado é descartado logo em
seguida (só o `history` importa aqui).

In [ ]:
import torch
from act_lang.training.checkpoints import load_checkpoint

device = torch.device("cpu")
model, _optimizer = train_script.build_model_and_optimizer(cfg, device)

checkpoint_dir = Path(CHECKPOINT_DIR)
_, history = load_checkpoint(checkpoint_dir / "last_checkpoint.pt", model, device=device)

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss total (recon + kl_weight*KL)")

axes[1].plot(history["train_recon"], label="train (z~q)")
axes[1].plot(history["val_recon"], label="val (z=mu)")
axes[1].plot(history["val_recon_z0"], label="val (z=0)", linestyle="--")
axes[1].set_title("Recon L1 — z0 é a métrica de seleção")

axes[2].plot(history["train_kld"], label="train")
axes[2].plot(history["val_kld"], label="val")
axes[2].set_title("kld_raw")

axes[3].plot(history["train_mu_abs_mean"], label="train")
axes[3].plot(history["val_mu_abs_mean"], label="val")
axes[3].axhline(0, color="gray", linestyle=":", linewidth=1)
axes[3].set_title("|mu| médio — perto de 0 = posterior colapsado")

for ax in axes:
    ax.set_xlabel("época"); ax.legend()
fig.suptitle(f"{BACKBONE_TYPE} + {FUSION_TYPE}")
plt.tight_layout(); plt.show()